In [1]:
import pandas as pd
import os
import warnings
from src.config import *

In [2]:
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

In [3]:
region_list = ['강릉', '대관령']

In [4]:
for n, region in enumerate(region_list):
    # 현재 경로 확인
    data_folder = RAW_DATA_DIR + 'spi/' + region + '/'
    print(data_folder)

    # 파일명 확인
    file_names = os.listdir(data_folder)
    print(file_names[:1])

    # 데이터 폴더 속 파일 각각의 경로 확인
    file_paths = []
    for file_name in file_names:
        file_paths.append(os.path.join(data_folder, file_name))
    print(file_paths[:1])

    # csv 파일들을 DataFrame으로 불러와서 concat
    concat_df = pd.DataFrame()
    for file_path in file_paths:
        df = pd.read_csv(file_path, encoding='cp949')

        # 일시, SPI1, SPI2, SPI3, SPI4만 추출
        selected_df = df[['일시', 'SPI1', 'SPI2', 'SPI3', 'SPI4']]

        concat_df = pd.concat([concat_df, selected_df])

    concat_df['일시'] = pd.to_datetime(concat_df['일시'])
    concat_df = concat_df.sort_values('일시').reset_index(drop=True) # os.listdir()은 파일의 순서를 보장하지 않기 때문에 정렬 필요

    # 보간법
    full_idx = pd.date_range('2002-01-01', '2025-12-31')
    concat_df = concat_df.set_index('일시').reindex(full_idx)

    concat_df[['SPI1','SPI2','SPI3','SPI4']] = (
        concat_df[['SPI1','SPI2','SPI3','SPI4']]
        .interpolate(method='linear')
        .bfill()
        .ffill()
    )

    concat_df.index.name = '일시'
    concat_df = concat_df.reset_index()

    print(f"reindex 후 행 수: {len(concat_df)}")  # 8766이어야 함
    print(f"NaN 잔존: {concat_df[['SPI1','SPI2','SPI3','SPI4']].isnull().sum().to_dict()}")
    

    # 지역별로 합쳐진 하나의 DataFrame을 csv로 export
    concat_df.to_csv(PROCESSED_DIR + 'spi_' + region_list[n] + '_2002_2025.csv', encoding='utf-8-sig', index=False)



/Users/jeongseok/Desktop/업무/2. 실측가뭄/4. 가뭄영향정보 생산 기술 개발/6. 실측가뭄_농산물 도매가격 예측모델_스크립트/raw_data/spi/강릉/
['ENV_SPI_105_DAY_2005_2005_2021.csv']
['/Users/jeongseok/Desktop/업무/2. 실측가뭄/4. 가뭄영향정보 생산 기술 개발/6. 실측가뭄_농산물 도매가격 예측모델_스크립트/raw_data/spi/강릉/ENV_SPI_105_DAY_2005_2005_2021.csv']
reindex 후 행 수: 8766
NaN 잔존: {'SPI1': 0, 'SPI2': 0, 'SPI3': 0, 'SPI4': 0}
/Users/jeongseok/Desktop/업무/2. 실측가뭄/4. 가뭄영향정보 생산 기술 개발/6. 실측가뭄_농산물 도매가격 예측모델_스크립트/raw_data/spi/대관령/
['ENV_SPI_100_DAY_2007_2007_2025.csv']
['/Users/jeongseok/Desktop/업무/2. 실측가뭄/4. 가뭄영향정보 생산 기술 개발/6. 실측가뭄_농산물 도매가격 예측모델_스크립트/raw_data/spi/대관령/ENV_SPI_100_DAY_2007_2007_2025.csv']
reindex 후 행 수: 8766
NaN 잔존: {'SPI1': 0, 'SPI2': 0, 'SPI3': 0, 'SPI4': 0}
